# Example 11.4.2 / 1: A fixed-charge nonlinear allocation

Companion notebook for **11.4.2 Mixed-Integer Nonlinear Models** in the AMPL lab Documentation, section "A fixed-charge nonlinear allocation". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## A fixed-charge nonlinear allocation

This small model selects projects and allocates production. Its separable quadratic operating cost is convex, while binary setup decisions make the overall problem discrete.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PLANTS;
param demand > 0;
param capacity {PLANTS} > 0;
param fixed_cost {PLANTS} >= 0;
param quad_cost {PLANTS} > 0;

var Open {PLANTS} binary;
var Produce {PLANTS} >= 0;

minimize TotalCost:
    sum {p in PLANTS}
        (fixed_cost[p] * Open[p] + quad_cost[p] * Produce[p]^2);

subject to MeetDemand:
    sum {p in PLANTS} Produce[p] >= demand;

subject to CapacityLink {p in PLANTS}:
    Produce[p] <= capacity[p] * Open[p];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PLANTS"] = ["north", "south"]
ampl.param["demand"] = 7
ampl.param["capacity"] = {"north": 6, "south": 8}
ampl.param["fixed_cost"] = {"north": 5, "south": 8}
ampl.param["quad_cost"] = {"north": 1, "south": 0.5}

In [5]:
# run.py
ampl.solve(solver="bonmin", verbose=False)
print(ampl.solve_result)
print(ampl.var["Open"].to_dict())
print(ampl.var["Produce"].to_dict())

solved
{'north': 1, 'south': 1}
{'north': 2.3333333100015197, 'south': 4.6666666200006235}


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PLANTS := north south;
param demand := 7;
param capacity := north 6 south 8;
param fixed_cost := north 5 south 8;
param quad_cost := north 1 south 0.5;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver bonmin;
option bonmin_options 'bonmin.bb_log_level=0 bonmin.nlp_log_level=0 print_level=0';
option solver_msg 0;
solve;
display Open, Produce, TotalCost;

Bonmin 1.8.9 using Cbc 2.10.8 and Ipopt 3.12.13
bonmin: bonmin.bb_log_level=0
bonmin.nlp_log_level=0
print_level=0


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt
******************************************************************************


 	"Finished"
:     Open   Produce    :=
north   1    2.33333
south   1    4.66667
;

TotalCost = 29.3333

